# Analisis Regresi OLS: Belanja Modal Pendidikan & Kesehatan terhadap IPM

**Data Panel 97 Kota di Indonesia (2014–2024)**

---

## Latar Belakang

Belanja modal pendidikan dan kesehatan merupakan investasi pemerintah daerah untuk meningkatkan
kualitas sumber daya manusia. Notebook ini menguji pengaruh kedua jenis belanja tersebut terhadap
**Indeks Pembangunan Manusia (IPM)** menggunakan **regresi linier berganda (Ordinary Least Squares)**.

## Model Penelitian

$$\text{IPM} = \beta_0 + \beta_1 \cdot \text{bm\_pendidikan} + \beta_2 \cdot \text{bm\_kesehatan} + \varepsilon$$

| Simbol | Variabel | Keterangan |
|---|---|---|
| $Y$ | `IPM` | Indeks Pembangunan Manusia |
| $X_1$ | `bm_pendidikan` | Belanja modal fungsi pendidikan |
| $X_2$ | `bm_kesehatan` | Belanja modal fungsi kesehatan |

## Hipotesis

- **H1**: Belanja modal pendidikan berpengaruh positif terhadap IPM.
- **H2**: Belanja modal kesehatan berpengaruh positif terhadap IPM.

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 2. Memuat Data

Dataset panel fiskal 97 kota dibaca langsung dari repositori GitHub publik.

In [2]:
URL_DATA = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv"

try:
    df = pd.read_csv(URL_DATA)
except Exception:
    df = pd.read_csv("data/dataset_fiskal_97_hipoFull.csv")

df = df[["Kota", "Tahun", "bm_pendidikan", "bm_kesehatan", "IPM"]].dropna()

print(f"Jumlah observasi : {len(df)}")
print(f"Jumlah kota      : {df['Kota'].nunique()}")
print(f"Rentang tahun    : {df['Tahun'].min()} - {df['Tahun'].max()}")
df.head()

Jumlah observasi : 1067
Jumlah kota      : 97
Rentang tahun    : 2014 - 2024


,Kota,Tahun,bm_pendidikan,bm_kesehatan,IPM
0,Kota Banda Aceh,2014,143.6400,77.6200,65.3100
1,Kota Banda Aceh,2015,287.6800,138.6600,64.9900
2,Kota Banda Aceh,2016,233.0000,121.8100,71.0000
3,Kota Banda Aceh,2017,199.6600,124.0800,83.4900
4,Kota Banda Aceh,2018,89.0000,108.4000,73.5700


## 3. Statistik Deskriptif & Korelasi

In [3]:
kolom = ["bm_pendidikan", "bm_kesehatan", "IPM"]

print("Statistik Deskriptif:")
display(df[kolom].describe().round(2))

print("\nMatriks Korelasi:")
display(df[kolom].corr().round(3))

Statistik Deskriptif:


,bm_pendidikan,bm_kesehatan,IPM
count,"1,067.0000","1,067.0000","1,067.0000"
mean,173.7900,175.7800,72.3800
std,73.3500,72.8300,7.2100
min,51.1600,50.8000,60.0100
25%,109.6200,109.7500,66.0800
50%,175.6700,178.7000,72.4200
75%,237.5700,239.1200,78.7600
max,299.9300,299.8500,84.9500



Matriks Korelasi:


,bm_pendidikan,bm_kesehatan,IPM
bm_pendidikan,1.0000,0.0270,0.0240
bm_kesehatan,0.0270,1.0000,-0.0740
IPM,0.0240,-0.0740,1.0000


## 4. Estimasi Regresi OLS

Model diestimasi dengan `statsmodels`. Ringkasan output menampilkan koefisien,
*standard error*, statistik uji, serta ukuran kecocokan model (R² dan F-statistic).

In [4]:
model = smf.ols("IPM ~ bm_pendidikan + bm_kesehatan", data=df).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    IPM   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     3.331
Date:                Tue, 13 Oct 2026   Prob (F-statistic):             0.0361
Time:                        12:09:28   Log-Likelihood:                -3618.2
No. Observations:                1067   AIC:                             7242.
Df Residuals:                    1064   BIC:                             7257.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        73.2347      0.768     95.382

### Tabel Koefisien

In [5]:
tabel = pd.DataFrame({
    "Koefisien": model.params,
    "Std. Error": model.bse,
    "t-statistik": model.tvalues,
    "p-value": model.pvalues,
})
tabel["Signifikansi"] = np.where(
    model.pvalues < 0.01, "***",
    np.where(model.pvalues < 0.05, "**", np.where(model.pvalues < 0.10, "*", "")),
)
display(tabel.round(4))

print(f"R-squared      : {model.rsquared:.4f}")
print(f"Adj. R-squared : {model.rsquared_adj:.4f}")
print(f"F-statistic    : {model.fvalue:.4f}   (p-value = {model.f_pvalue:.4f})")

,Koefisien,Std. Error,t-statistik,p-value,Signifikansi
Intercept,73.2347,0.7678,95.3822,0.0000,***
bm_pendidikan,0.0026,0.0030,0.8625,0.3886,
bm_kesehatan,-0.0074,0.0030,-2.4551,0.0142,**


R-squared      : 0.0062
Adj. R-squared : 0.0044
F-statistic    : 3.3308   (p-value = 0.0361)


## 5. Visualisasi

Dua grafik sebaran (*scatter*) dengan garis regresi untuk melihat arah hubungan
masing-masing variabel bebas terhadap IPM, serta diagram batang nilai koefisien model.

In [6]:
pasangan = [
    ("bm_pendidikan", "Belanja Modal Pendidikan"),
    ("bm_kesehatan", "Belanja Modal Kesehatan"),
]

for kolom_x, judul in pasangan:
    # garis regresi sederhana (bivariat) untuk keperluan visualisasi
    garis = np.polyfit(df[kolom_x], df["IPM"], 1)
    x_garis = np.linspace(df[kolom_x].min(), df[kolom_x].max(), 100)

    fig = px.scatter(
        df, x=kolom_x, y="IPM", opacity=0.5,
        labels={kolom_x: judul, "IPM": "IPM"},
        title=f"<b>Sebaran {judul} vs IPM</b>",
    )
    fig.add_trace(go.Scatter(
        x=x_garis, y=np.polyval(garis, x_garis), mode="lines",
        name="Garis Regresi", line=dict(color="#E65100", width=3),
    ))
    fig.update_traces(marker=dict(color="#2E7D32", size=6), selector=dict(mode="markers"))
    fig.update_layout(template="plotly_white", height=470, showlegend=False)
    fig.show()

### Koefisien Regresi OLS

In [7]:
koef = pd.DataFrame({
    "Variabel": ["Belanja Modal Pendidikan", "Belanja Modal Kesehatan"],
    "Koefisien": [model.params["bm_pendidikan"], model.params["bm_kesehatan"]],
})

fig = px.bar(
    koef, x="Variabel", y="Koefisien", text_auto=".4f",
    color="Variabel",
    color_discrete_sequence=["#1E88E5", "#E65100"],
    title="<b>Nilai Koefisien Regresi OLS</b>",
)
fig.update_layout(template="plotly_white", height=440, showlegend=False)
fig.show()

## 6. Kesimpulan

**Persamaan regresi terestimasi:**

$$\widehat{\text{IPM}} = 73{,}2347 + 0{,}002592 \cdot X_1 - 0{,}007432 \cdot X_2$$

| Variabel | Koefisien | p-value | Keputusan (alpha = 5%) |
|---|---|---|---|
| Belanja Modal Pendidikan | +0,002592 | 0,3886 | Tidak signifikan |
| Belanja Modal Kesehatan | −0,007432 | 0,0142 | Signifikan (arah negatif) |

- **R² = 0,0062** → model hanya menjelaskan **0,62%** variasi IPM.
- **F-statistic = 3,331 (p = 0,0361)** → secara simultan kedua variabel berpengaruh signifikan.

**Interpretasi:**

1. **Belanja modal pendidikan tidak berpengaruh signifikan** terhadap IPM (p = 0,3886).
2. **Belanja modal kesehatan berpengaruh signifikan namun negatif** (p = 0,0142). Hal ini tidak
   berarti belanja kesehatan merugikan IPM, melainkan mencerminkan **alokasi berbasis kebutuhan**
   (kota dengan IPM rendah menganggarkan belanja modal kesehatan lebih besar) dan **efek jangka
   panjang** belanja modal yang baru terasa beberapa tahun kemudian.
3. **Daya penjelas model rendah (0,62%)** → IPM lebih banyak ditentukan faktor lain di luar
   belanja modal layanan dasar, seperti belanja operasional, kualitas SDM, dan kondisi sosial-ekonomi.